# 🎙️ AI4Bharat IndicConformer ASR (Bengali) - Test & Inference Notebook

This notebook demonstrates loading, preprocessing audio, and running speech-to-text inference with **AI4Bharat IndicConformer Bengali** (`ai4bharat/indicconformer_stt_bn_hybrid_ctc_rnnt_large`).

## 1. Imports and Device Setup

In [1]:
import os
import subprocess
from pathlib import Path
import torch
import torchaudio
import torchaudio.functional as F
import soundfile as sf
from huggingface_hub import hf_hub_download
import nemo.collections.asr as nemo_asr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Compute Device: {device}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")
print(f"[INFO] PyTorch Version: {torch.__version__}")

c:\Users\subho\OneDrive\Documents\VoiceLLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


[INFO] Compute Device: cuda
[INFO] GPU Model: NVIDIA GeForce RTX 2050
[INFO] PyTorch Version: 2.6.0+cu124


## 2. Download Model Checkpoint from Hugging Face Hub

In [2]:
MODEL_REPO = "ai4bharat/indicconformer_stt_bn_hybrid_ctc_rnnt_large"
MODEL_FILENAME = "indicconformer_stt_bn_hybrid_rnnt_large.nemo"

print(f"[INFO] Downloading '{MODEL_FILENAME}' from '{MODEL_REPO}'...")
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILENAME
)
print(f"[SUCCESS] Checkpoint downloaded at: {model_path}")

[INFO] Downloading 'indicconformer_stt_bn_hybrid_rnnt_large.nemo' from 'ai4bharat/indicconformer_stt_bn_hybrid_ctc_rnnt_large'...
[SUCCESS] Checkpoint downloaded at: C:\Users\subho\.cache\huggingface\hub\models--ai4bharat--indicconformer_stt_bn_hybrid_ctc_rnnt_large\snapshots\15a1cd06245262b914f27ed445a604b7b278d187\indicconformer_stt_bn_hybrid_rnnt_large.nemo


## 3. Load IndicConformer ASR Model

In [3]:
def load_indic_conformer_model(nemo_file_path: str, target_device: torch.device):
    """Loads AI4Bharat IndicConformer model cleanly."""
    print(f"[INFO] Restoring model from {nemo_file_path} on {target_device}...")
    model = nemo_asr.models.EncDecHybridRNNTCTCBPEModel.restore_from(
        restore_path=nemo_file_path,
        map_location=target_device,
        strict=False
    )
    model.freeze()
    model.eval()
    print("[SUCCESS] IndicConformer model successfully loaded!")
    return model

# Load model
asr_model = load_indic_conformer_model(model_path, device)

[INFO] Restoring model from C:\Users\subho\.cache\huggingface\hub\models--ai4bharat--indicconformer_stt_bn_hybrid_ctc_rnnt_large\snapshots\15a1cd06245262b914f27ed445a604b7b278d187\indicconformer_stt_bn_hybrid_rnnt_large.nemo on cuda...
[NeMo I 2026-09-01 18:59:06 mixins:77] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-01 18:59:06 mixins:234] Tokenizer SentencePieceTokeni

[NeMo W 2026-09-01 18:59:20 rnnt_models:63] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/manifests/nemo/vistaar_v3/train/train_bengali.json
    sample_rate: 16000
    batch_size: 8
    shuffle: false
    num_workers: 16
    pin_memory: true
    max_duration: 30.0
    min_duration: 0.2
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    is_concat: true
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1.5
    return_language_id: true
    
[NeMo W 2026-09-01 18:59:20 rnnt_models:63] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid confi

[NeMo I 2026-09-01 18:59:25 rnnt_models:83] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-01 18:59:25 rnnt_models:231] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-01 18:59:27 rnnt_models:231] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-01 18:59:28 modelPT:489] Model EncDecHybridRNNTCTCBPEModel was successfully restored from C:\Users\subho\.cache\huggingface\hub\models--ai4bharat--indicconformer_stt_bn_hybrid_ctc_rnnt_large\snapshots\15a1cd06245262b914f27ed445a604b7b278d187\indicconformer_stt_bn_hybrid_rnnt_large.nemo.
[SUCCESS] IndicConformer model successfully loaded!


## 4. Audio Preprocessing Utility (16 kHz Mono Standardizer)

Converts any input audio format (`.m4a`, `.mp3`, `.ogg`, `.flac`, `.wav`) to the required 16 kHz mono WAV format using `ffmpeg` / `librosa` / `torchaudio`.

In [ ]:
def preprocess_audio_for_asr(input_path: str, output_path: str = "audio_for_inference.wav", target_sr: int = 16000) -> str:
    """Converts any audio file (.m4a, .mp3, .wav, etc.) to 16kHz mono WAV format."""
    path = Path(input_path)
    if not path.exists():
        raise FileNotFoundError(f"Audio file does not exist: {input_path}")
    
    # 1. Try ffmpeg (fastest, supports .m4a / AAC / MP3 / WAV without libsndfile limitations)
    try:
        cmd = ["ffmpeg", "-y", "-i", str(path), "-ar", str(target_sr), "-ac", "1", output_path]
        res = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if res.returncode == 0 and os.path.exists(output_path):
            info = sf.info(output_path)
            print(f"[INFO] Preprocessed '{input_path}' -> '{output_path}' via ffmpeg (16kHz mono, duration: {info.duration:.2f}s)")
            return output_path
    except Exception:
        pass
    
    # 2. Fallback to librosa
    try:
        import librosa
        y, _ = librosa.load(str(path), sr=target_sr, mono=True)
        sf.write(output_path, y, target_sr)
        print(f"[INFO] Preprocessed '{input_path}' -> '{output_path}' via librosa (16kHz mono, duration: {len(y)/target_sr:.2f}s)")
        return output_path
    except Exception:
        pass
    
    # 3. Fallback to torchaudio
    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != target_sr:
        waveform = F.resample(waveform, orig_freq=sr, new_freq=target_sr)
    
    audio_np = waveform.squeeze().cpu().numpy()
    sf.write(output_path, audio_np, target_sr)
    duration = len(audio_np) / target_sr
    print(f"[INFO] Preprocessed '{input_path}' -> '{output_path}' via torchaudio (16kHz mono, duration: {duration:.2f}s)")
    return output_path

# Locate test audio (vocaltest.m4a or fallback to synthetic sample)
candidates = [Path("vocaltest.m4a"), Path("tests/vocaltest.m4a"), Path("../tests/vocaltest.m4a")]
demo_audio = next((str(p) for p in candidates if p.exists()), None)

if not demo_audio:
    print("[INFO] 'vocaltest.m4a' not found. Generating synthetic tone audio for pipeline validation...")
    sr = 16000
    t = torch.linspace(0, 2.0, sr * 2)
    synthetic = (0.25 * torch.sin(2 * 3.14159 * 440.0 * t)).numpy()
    demo_audio = "sample_synthetic_audio.wav"
    sf.write(demo_audio, synthetic, sr)

clean_audio_path = preprocess_audio_for_asr(demo_audio, "audio_for_inference.wav")

## 5. Transcribe Audio (CTC & RNN-T Decoding Modes)

- **CTC Decoder (`decoder='ctc'`):** Fast non-autoregressive decoding.
- **RNN-T Decoder (`decoder='rnnt'`):** Autoregressive decoding.

In [7]:
def transcribe_audio(audio_path: str, decoder: str = "ctc") -> str:
    """Transcribes a 16kHz mono WAV file using the chosen decoder."""
    if decoder not in ["ctc", "rnnt"]:
        raise ValueError("Decoder must be 'ctc' or 'rnnt'")
    
    asr_model.change_decoding_strategy(decoder_type=decoder)
    with torch.no_grad():
        predictions = asr_model.transcribe([audio_path], batch_size=1)
    
    if isinstance(predictions, list) and len(predictions) > 0:
        pred = predictions[0]
        return pred.text if hasattr(pred, 'text') else str(pred)
    return str(predictions)

# 1. CTC Transcription
print("=== [1] CTC Mode Inference ===")
transcript_ctc = transcribe_audio(clean_audio_path, decoder="ctc")
print(f"CTC Transcript: {transcript_ctc}")

# 2. RNN-T Transcription
print("\n=== [2] RNN-T Mode Inference ===")
transcript_rnnt = transcribe_audio(clean_audio_path, decoder="rnnt")
print(f"RNN-T Transcript: {transcript_rnnt}")

=== [1] CTC Mode Inference ===
[NeMo I 2026-08-27 01:21:16 3867558774:6] No `decoding_cfg` passed when changing decoding strategy, using internal config
[NeMo I 2026-08-27 01:21:16 3867558774:6] Changed decoding strategy of the CTC decoder to 
    strategy: greedy
    preserve_alignments: null
    compute_timestamps: null
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    ctc_timestamp_type: all
    batch_dim_index: 0
    greedy:
      preserve_alignments: false
      compute_timestamps: false
      preserve_frame_confidence: false
      confidence_method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
      ngram_lm_model: null
      ngram_lm_alpha: 0.0
      boosting_tree:
        model_path: null
        key_phrases_file: null
        key_phrases_list: null
        key_phrase_items_list: null
        context_score: 1.0
        depth_scal

[NeMo W 2026-08-27 01:21:16 dataloader:343] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-27 01:21:16 dataloader:349] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s][NeMo W 2026-08-27 01:21:17 common:1381] CTC decoding strategy 'greedy' is slower than 'greedy_batch', which implements the same exact interface. Consider changing your strategy to 'greedy_batch' for a free performance improvement.
Transcribing: 1it [00:00,  1.04it/s]

CTC Transcript: িস ট এখন আমি বাংলাতে আছিড কথা বলে যাচ্ছি এরপরে আমি হিন্দিতে চাঞ্জ যাবোো আি ম হিন্দি ম বার্তালপ করহো

=== [2] RNN-T Mode Inference ===
[NeMo I 2026-08-27 01:21:17 3867558774:6] No `decoding_cfg` passed when changing decoding strategy, using internal config
[NeMo I 2026-08-27 01:21:17 rnnt_models:231] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo I 2026-08-27 01:21:17 3867558774:6] Changed decoding strategy of the RNNT decoder to 
    model_type: rnnt
    strategy: greedy_batch
    compute_hypothesis_token_set: false
    preserve_alignments: null
    tdt_include_token_duration: null
    confidence_cfg:
      preserve_frame_confidence: false
      preserve_token_confidence: false
      preserve_word_confidence: false
      exclude_blank: true
      aggregation: min
      tdt_include_duration: false
      method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
    fused_batch_size: null
    compute_timestamps: null
    compute_langs: false
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    rnnt_timestamp_type: all
    greedy:
      max_symbols_per_step: 10
      preserve_alignments: false
      preserve_frame_confidence: false
      tdt_include_token_duration: false
      tdt_inc

[NeMo W 2026-08-27 01:21:17 dataloader:343] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-27 01:21:18 dataloader:349] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:03,  3.28s/it]

RNN-T Transcript: द ᱵᱟᱝेश ಲ உள்ளোন్వೇಳ पण पण पण पण ᱵᱟᱝहीని ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददददᱮᱮ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲऒऒऒऒऒऒऒऒऒऒराराराराराराराराराराानराानराराোনোনোনোন؛োন మ यांॐ यांॐ यां؛োন؛োন এক चಿದ್ದಾರೆ यांਿੱ यां এ यां এক؛ګऎګ்்்்ژ்्टानानानानानानঐঐঐ खौ खौ खौ खौ  ꯡ  लीली ꯄꯨꯋꯥेली हु राजకాानऽान घऒ଼ पण଼ पण଼ पण଼ पण మ మ మಿದ್ದಾರೆ ꯃꯍꯥꯛ একೂ ꯄꯨꯋꯥ؛णार ꯄꯨꯋꯥब्ಾಜ؛ब्णार এক ꯄꯨꯋꯥ؛؛ान మٟ؛अअअअअअ मु मु मु मु मु मु मु मु ꯄꯨꯋꯥ ꯄꯨꯋꯥ मु मु मु मु मु मु मु मु मु मुअअതതതതതതതത कारान ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ ಲ଼पि଼଼଼଼଼଼଼଼଼଼଼଼଼଼଼଼଼଼६ুর्टুর्ट्ट्टर्६६଼଼଼଼଼଼଼଼଼଼ अस଼ अस଼଼଼଼଼଼଼ॠॠરરરરરરર ജ    खखखखखख खौ खौ खौ खौ खौ खौ खौ खौ खौ खौ खौ खौ खौ ಲ ಲ ಲ ಲ ಲ ಲ ಲ्ह्ह्ह्ह ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ మली हु ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ ꯃꯍꯥꯛ మली ꯑꯗꯨ ಲानानानानानानानानङानङानङानङानङानاق ಲاقاقاق